Important Note: This notebook contains the LSEG Workspace data collection pipeline used to produce the enrichment files in data/lseg/. A condensed version of this code is also included in the main report (ST115_Final_Report.ipynb) under Section 2.3 for marking purposes. This notebook is provided as a supplementary file for reference and completeness. Re-running it requires LSEG Workspace to be installed and running in the background. All outputs have already been saved to data/lseg/ and are loaded directly in the main report.

In [20]:
!pip install refinitiv-data

In [22]:
import refinitiv.data as rd
import pandas as pd
import os
import time

rd.open_session()

COMBINED_CSV = "../data/edgar/combined_13f_2006_2025.csv"
OUTPUT_DIR = "../data/lseg"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "cusip_ric_mapping.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(COMBINED_CSV, dtype={"cusip": str})
cusips = sorted(df["cusip"].dropna().unique())
cusips = [c for c in cusips if len(c) == 9]
print(f"{len(cusips)} unique 9-char CUSIPs to map")

11974 unique 9-char CUSIPs to map


In [28]:
# CUSIP to RIC Mapping and Sector Classification

FIELDS = [
    "TR.RIC",
    "TR.CommonName",
    "TR.TRBCEconomicSector",
    "TR.TRBCBusinessSector",
    "TR.TRBCIndustryGroup",
    "TR.TRBCIndustry",
]
BATCH = 200

frames = []

for i in range(0, len(cusips), BATCH):
    batch = cusips[i : i + BATCH]
    print(f"  Batch {i+1}–{i+len(batch)} / {len(cusips)}...", end=" ")

    try:
        result = rd.get_data(batch, fields=FIELDS)
        if result is not None and not result.empty:
            frames.append(result)
            matched = result["RIC"].notna().sum() if "RIC" in result.columns else 0
            print(f"{matched}/{len(batch)} matched")
        else:
            print("empty response")
    except Exception as e:
        print(f"error: {e}")

    time.sleep(0.3)

raw_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nRows returned: {len(raw_df)}")
if not raw_df.empty:
    print(f"With RIC: {raw_df['RIC'].notna().sum()}")
raw_df.head(10)

  Batch 1–200 / 11974... 200/200 matched
  Batch 201–400 / 11974... 200/200 matched
  Batch 401–600 / 11974... 200/200 matched
  Batch 601–800 / 11974... 200/200 matched
  Batch 801–1000 / 11974... 200/200 matched
  Batch 1001–1200 / 11974... 200/200 matched
  Batch 1201–1400 / 11974... 201/200 matched
  Batch 1401–1600 / 11974... 200/200 matched
  Batch 1601–1800 / 11974... 200/200 matched
  Batch 1801–2000 / 11974... 200/200 matched
  Batch 2001–2200 / 11974... 200/200 matched
  Batch 2201–2400 / 11974... 200/200 matched
  Batch 2401–2600 / 11974... 200/200 matched
  Batch 2601–2800 / 11974... 200/200 matched
  Batch 2801–3000 / 11974... 200/200 matched
  Batch 3001–3200 / 11974... 200/200 matched
  Batch 3201–3400 / 11974... 200/200 matched
  Batch 3401–3600 / 11974... 200/200 matched
  Batch 3601–3800 / 11974... 161/200 matched
  Batch 3801–4000 / 11974... 157/200 matched
  Batch 4001–4200 / 11974... 200/200 matched
  Batch 4201–4400 / 11974... 200/200 matched
  Batch 4401–4600 / 1

,Instrument,RIC,Company Common Name,TRBC Economic Sector Name,TRBC Business Sector Name,TRBC Industry Group Name,TRBC Industry Name
0,000307108,AACH.PK^L20,AAC Holdings Inc,Healthcare,Healthcare Services & Equipment,Healthcare Providers & Services,Healthcare Facilities & Services
1,00032Q104,WHWK.OQ,Whitehawk Therapeutics Inc,Healthcare,Pharmaceuticals & Medical Research,Pharmaceuticals,Pharmaceuticals
2,000360206,AAON.OQ,Aaon Inc,Industrials,Industrial Goods,"Machinery, Tools, Heavy Vehicles, Trains & Ships",Electrical Components & Equipment
3,000361105,AIR.N,AAR Corp,Industrials,Industrial Goods,Aerospace & Defense,Aerospace & Defense
4,000375204,ABBNY.PK,Abb Ltd,Industrials,Industrial Goods,"Machinery, Tools, Heavy Vehicles, Trains & Ships",Heavy Electrical Equipment
5,000380204,ABCM.OQ^L23,Abcam Ltd,Healthcare,Pharmaceuticals & Medical Research,Biotechnology & Medical Research,Biotechnology & Medical Research
6,00080S101,,<NA>,<NA>,<NA>,<NA>,<NA>
7,00081T108,ACCO.N,ACCO Brands Corp,Industrials,Industrial & Commercial Services,Professional & Commercial Services,Business Support Supplies
8,000868109,ACNB.OQ,ACNB Corp,Financials,Banking & Investment Services,Banking Services,Banks
9,00086T103,ACMR.OQ^K11,AC Moore Arts and Crafts Inc,Consumer Cyclicals,Retailers,Specialty Retailers,Miscellaneous Specialty Retailers


In [30]:
# Cleaning up and saving mapping
col_rename = {
    "Instrument": "cusip",
    "RIC": "ric",
    "Common Name": "company_name",
    "TRBC Economic Sector Name": "trbc_economic_sector",
    "TRBC Business Sector Name": "trbc_business_sector",
    "TRBC Industry Group Name": "trbc_industry_group",
    "TRBC Industry Name": "trbc_industry",
}
# Handling raw field names as fallback
col_rename_alt = {
    "TR.RIC": "ric",
    "TR.CommonName": "company_name",
    "TR.TRBCEconomicSector": "trbc_economic_sector",
    "TR.TRBCBusinessSector": "trbc_business_sector",
    "TR.TRBCIndustryGroup": "trbc_industry_group",
    "TR.TRBCIndustry": "trbc_industry",
}

mapping = raw_df.rename(columns=col_rename).rename(columns=col_rename_alt)

# Keeping the first one for cases where CUSIPs resolve to multiple RICs
mapping = mapping.drop_duplicates(subset="cusip", keep="first")

# Adding unmatched CUSIPs so gaps are visible
matched_cusips = set(mapping["cusip"].dropna())
unmatched = [c for c in cusips if c not in matched_cusips]
if unmatched:
    unmatched_df = pd.DataFrame({"cusip": unmatched})
    mapping = pd.concat([mapping, unmatched_df], ignore_index=True)

mapping = mapping.sort_values("cusip").reset_index(drop=True)
mapping.to_csv(OUTPUT_FILE, index=False)

total = len(mapping)
with_ric = mapping["ric"].notna().sum()
with_sector = mapping["trbc_economic_sector"].notna().sum()
print(f"Saved {OUTPUT_FILE}")
print(f"  {total} CUSIPs total")
print(f"  {with_ric} with RIC ({with_ric/total:.1%})")
print(f"  {with_sector} with TRBC sector ({with_sector/total:.1%})")
print(f"  {total - with_ric} unmatched (delisted/expired)")
mapping.head(10)

Saved ../data/lseg/cusip_ric_mapping.csv
  11974 CUSIPs total
  11801 with RIC (98.6%)
  9700 with TRBC sector (81.0%)
  173 unmatched (delisted/expired)


,cusip,ric,Company Common Name,trbc_economic_sector,trbc_business_sector,trbc_industry_group,trbc_industry
0,000307108,AACH.PK^L20,AAC Holdings Inc,Healthcare,Healthcare Services & Equipment,Healthcare Providers & Services,Healthcare Facilities & Services
1,00032Q104,WHWK.OQ,Whitehawk Therapeutics Inc,Healthcare,Pharmaceuticals & Medical Research,Pharmaceuticals,Pharmaceuticals
2,000360206,AAON.OQ,Aaon Inc,Industrials,Industrial Goods,"Machinery, Tools, Heavy Vehicles, Trains & Ships",Electrical Components & Equipment
3,000361105,AIR.N,AAR Corp,Industrials,Industrial Goods,Aerospace & Defense,Aerospace & Defense
4,000375204,ABBNY.PK,Abb Ltd,Industrials,Industrial Goods,"Machinery, Tools, Heavy Vehicles, Trains & Ships",Heavy Electrical Equipment
5,000380204,ABCM.OQ^L23,Abcam Ltd,Healthcare,Pharmaceuticals & Medical Research,Biotechnology & Medical Research,Biotechnology & Medical Research
6,00080S101,,<NA>,<NA>,<NA>,<NA>,<NA>
7,00081T108,ACCO.N,ACCO Brands Corp,Industrials,Industrial & Commercial Services,Professional & Commercial Services,Business Support Supplies
8,000868109,ACNB.OQ,ACNB Corp,Financials,Banking & Investment Services,Banking Services,Banks
9,00086T103,ACMR.OQ^K11,AC Moore Arts and Crafts Inc,Consumer Cyclicals,Retailers,Specialty Retailers,Miscellaneous Specialty Retailers


In [36]:
# Historical Quarterly Prices

import refinitiv.data as rd
import pandas as pd
import time
import os

OUTPUT_DIR = "../data/lseg"
MAPPING_FILE = os.path.join(OUTPUT_DIR, "cusip_ric_mapping.csv")
PRICES_FILE = os.path.join(OUTPUT_DIR, "quarterly_prices.csv")

# Loading mapped RICs
mapping = pd.read_csv(MAPPING_FILE, dtype=str)
rics = sorted(mapping["ric"].dropna().unique().tolist())
print(f"{len(rics)} unique RICs to price")

# Getting quarter-end dates 2006–2025
qe_dates = []
for year in range(2006, 2026):
    for month in ["03-31", "06-30", "09-30", "12-31"]:
        qe_dates.append(f"{year}-{month}")

BATCH = 300
all_frames = []

for i in range(0, len(rics), BATCH):
    batch = rics[i : i + BATCH]
    print(f"  Batch {i+1}–{i+len(batch)} / {len(rics)}...", end=" ")

    try:
        result = rd.get_history(
            universe=batch,
            fields=["TRDPRC_1"],
            interval="monthly",
            start="2006-01-01",
            end="2025-12-31",
        )

        if result is not None and not result.empty:
            # get_history returns a DatetimeIndex; filter to quarter-end months
            df = result.copy()
            df.index = pd.to_datetime(df.index)
            df = df[df.index.month.isin([3, 6, 9, 12])]

            # Keep only the last trading day per month (the actual quarter-end close)
            df = df.groupby([df.index.year, df.index.month]).tail(1)

            all_frames.append(df)
            ric_count = df.shape[1] if df.ndim == 2 else 1
            print(f"{ric_count} RICs, {len(df)} dates")
        else:
            print("empty")

    except Exception as e:
        print(f"error: {e}")

    time.sleep(0.5)

print(f"\nCombining {len(all_frames)} batches...")

9554 unique RICs to price
  Batch 1–300 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 80 dates
  Batch 301–600 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 80 dates
  Batch 601–900 / 9554... 

...

300 RICs, 80 dates
  Batch 8401–8700 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 80 dates
  Batch 8701–9000 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 80 dates
  Batch 9001–9300 / 9554... error: Error code -1 | 'headers'
  Batch 9301–9554 / 9554... error: Error code -1 | No data to return, please check errors: ERROR: No successful response.
(429, Too many requests, please try again later)

Combining 29 batches...


In [48]:
#Retrying failed batches
failed_batches = [
    rics[6600:6900],
    rics[9000:9300],
    rics[9300:9554],
]

for j, batch in enumerate(failed_batches):
    print(f"  Retry {j+1}/3 ({len(batch)} RICs)...", end=" ")
    try:
        result = rd.get_history(
            universe=batch,
            fields=["TRDPRC_1"],
            interval="monthly",
            start="2006-01-01",
            end="2025-12-31",
        )
        if result is not None and not result.empty:
            df = result.copy()
            df.index = pd.to_datetime(df.index)
            df = df[df.index.month.isin([3, 6, 9, 12])]
            df = df.groupby([df.index.year, df.index.month]).tail(1)
            all_frames.append(df)
            print(f"{df.shape[1]} RICs, {len(df)} dates")
        else:
            print("empty")
    except Exception as e:
        print(f"error: {e}")
    time.sleep(2)

print(f"Total batches now: {len(all_frames)}")

  Retry 1/3 (300 RICs)... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 80 dates
  Retry 2/3 (300 RICs)... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 80 dates
  Retry 3/3 (254 RICs)... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


254 RICs, 80 dates
Total batches now: 32


In [50]:
# Reshaping and saving (streaming, one batch at a time)
import csv

header_written = False
row_count = 0
ric_set = set()

with open(PRICES_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ric", "date", "quarter", "close_price"])

    for idx, df in enumerate(all_frames):
        df = df.copy()
        df.index = pd.to_datetime(df.index)

        for col in df.columns:
            for dt, val in df[col].dropna().items():
                q = f"{dt.year}Q{(dt.month - 1) // 3 + 1}"
                writer.writerow([col, dt.strftime("%Y-%m-%d"), q, val])
                row_count += 1
                ric_set.add(col)

        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx+1}/{len(all_frames)} batches ({row_count} rows so far)")

print(f"\nSaved {PRICES_FILE}")
print(f"  {row_count} price observations")
print(f"  {len(ric_set)} unique RICs")

  Processed 10/32 batches (115251 rows so far)
  Processed 20/32 batches (233069 rows so far)
  Processed 30/32 batches (349707 rows so far)

Saved ../data/lseg/quarterly_prices.csv
  372803 price observations
  9449 unique RICs


In [52]:
# Point-in-Time Market Capitalisation

import refinitiv.data as rd
import pandas as pd
import csv
import time
import os

MAPPING_FILE = "../data/lseg/cusip_ric_mapping.csv"
MKTCAP_FILE = "../data/lseg/quarterly_market_cap.csv"

mapping = pd.read_csv(MAPPING_FILE, dtype=str)
rics = sorted(mapping["ric"].dropna().unique().tolist())
print(f"{len(rics)} RICs to fetch market cap for")

BATCH = 300
mktcap_frames = []

for i in range(0, len(rics), BATCH):
    batch = rics[i : i + BATCH]
    print(f"  Batch {i+1}–{i+len(batch)} / {len(rics)}...", end=" ")

    try:
        result = rd.get_data(
            universe=batch,
            fields=["TR.CompanyMarketCap", "TR.CompanyMarketCap.Date"],
            parameters={"SDate": "2006-01-01", "EDate": "2025-12-31", "Frq": "CQ"},
        )

        if result is not None and not result.empty:
            mktcap_frames.append(result)
            ric_count = result["Instrument"].nunique()
            print(f"{ric_count} RICs, {len(result)} rows")
        else:
            print("empty")
    except Exception as e:
        print(f"error: {e}")

    time.sleep(0.5)

print(f"\n{len(mktcap_frames)} batches collected")

9554 RICs to fetch market cap for
  Batch 1–300 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 21314 rows
  Batch 301–600 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 22262 rows
  Batch 601–900 / 9554... 

...

300 RICs, 22499 rows
  Batch 8701–9000 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 19655 rows
  Batch 9001–9300 / 9554... 

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


300 RICs, 22499 rows
  Batch 9301–9554 / 9554... error: Error code -1 | UDF Core request failed. Gateway Time-out Requested universes: ['WRN.TO', 'WS.N', 'WSBC.OQ', 'WSBCO.OQ', 'WSBF.OQ', 'WSC.A^F11', 'WSC.OQ', 'WSCWW.PK^F20', 'WSFS.OQ', 'WSII.OQ^G10', 'WSM.N', 'WSO.N', 'WSR.N', 'WST.N', 'WSTC.O^J17', 'WSUPW.PK', 'WT.N', 'WTB.L', 'WTBA.OQ', 'WTFC.OQ', 'WTI.N', 'WTM.N', 'WTRE.OQ^G21', 'WTRG.N', 'WTRHW.PK^C19', 'WTS.N', 'WTSLQ.PK^A16', 'WTTR.N', 'WTW.OQ', 'WU.N', 'WUBA.N^I20', 'WULF.OQ', 'WVB1.F^C10', 'WVE.OQ', 'WW.OQ', 'WWAV.K^D17', 'WWD.OQ', 'WWIN.PK^E08', 'WWR.A', 'WWW.N', 'WWY.N^J08', 'WX.N^L15', 'WY.N', 'WYE.N^J09', 'WYFI.OQ', 'WYNN.OQ', 'WYY.A', 'X.N^F25', 'XAIR.OQ', 'XAR.P', 'XBI.P', 'XBIT.OQ', 'XBKS.OQ^A18', 'XBPEW.OQ', 'XCRA.OQ^J18', 'XEC.N^J21', 'XEL.OQ', 'XELAW.PK^G22', 'XELB.OQ', 'XELLL.OQ', 'XENE.OQ', 'XENT.OQ^E22', 'XES.P', 'XFOR.OQ', 'XGN.OQ', 'XHB.P', 'XHE.P', 'XHR.N', 'XIDEQ.PK^E15', 'XIFR.N', 'XIV.OQ^B18', 'XL.N^I18', 'XLB.P', 'XLC.P', 'XLE.P', 'XLF.P', 'XLG.P', 'XLI.P'

In [54]:
#Retrying failed batch
batch = rics[9300:9554]
print(f"Retrying {len(batch)} RICs...", end=" ")

try:
    result = rd.get_data(
        universe=batch,
        fields=["TR.CompanyMarketCap", "TR.CompanyMarketCap.Date"],
        parameters={"SDate": "2006-01-01", "EDate": "2025-12-31", "Frq": "CQ"},
    )

    if result is not None and not result.empty:
        mktcap_frames.append(result)
        print(f"OK ({result['Instrument'].nunique()} RICs, {len(result)} rows)")
    else:
        print("empty response")
except Exception as e:
    print(f"error: {e}")

print("Done. Re-run the stream-to-CSV cell to regenerate quarterly_market_cap.csv")

Retrying 254 RICs... OK (254 RICs, 19056 rows)
Done. Re-run the stream-to-CSV cell to regenerate quarterly_market_cap.csv


/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [56]:
# Saving (streaming, one batch at a time)
row_count = 0
ric_set = set()

with open(MKTCAP_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ric", "date", "quarter", "market_cap"])

    for idx, df in enumerate(mktcap_frames):
        # Normalise column names (label vs raw field name)
        cols = df.columns.tolist()
        rename = {}
        for c in cols:
            cl = c.lower()
            if c == "Instrument":
                rename[c] = "ric"
            elif "date" in cl:
                rename[c] = "date"
            elif "market" in cl or "companym" in cl:
                rename[c] = "market_cap"
        df = df.rename(columns=rename)

        for _, row in df.iterrows():
            ric = row.get("ric")
            dt = row.get("date")
            mc = row.get("market_cap")

            if pd.isna(ric) or pd.isna(dt) or pd.isna(mc):
                continue

            dt = pd.to_datetime(dt)
            q = f"{dt.year}Q{(dt.month - 1) // 3 + 1}"
            writer.writerow([ric, dt.strftime("%Y-%m-%d"), q, mc])
            row_count += 1
            ric_set.add(ric)

        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx+1}/{len(mktcap_frames)} batches ({row_count} rows so far)")

print(f"\nSaved {MKTCAP_FILE}")
print(f"  {row_count} observations")
print(f"  {len(ric_set)} unique RICs")

  Processed 10/32 batches (155098 rows so far)
  Processed 20/32 batches (309725 rows so far)
  Processed 30/32 batches (463642 rows so far)

Saved ../data/lseg/quarterly_market_cap.csv
  493640 observations
  8774 unique RICs


In [58]:
# Data quality summary
import pandas as pd
import os

# Loading all datasets
holdings = pd.read_csv("../data/edgar/combined_13f_2006_2025.csv", dtype={"cusip": str})
mapping = pd.read_csv("../data/lseg/cusip_ric_mapping.csv", dtype={"cusip": str, "ric": str})
prices = pd.read_csv("../data/lseg/quarterly_prices.csv", dtype={"ric": str})
mktcap = pd.read_csv("../data/lseg/quarterly_market_cap.csv", dtype={"ric": str})

# Merging mapping onto holdings
ric_lookup = mapping[["cusip", "ric", "trbc_economic_sector"]].drop_duplicates(subset="cusip")
df = holdings.merge(ric_lookup, on="cusip", how="left")

# Flagging which RICs have price / mktcap data per quarter
price_keys = set(zip(prices["ric"], prices["quarter"]))
mktcap_keys = set(zip(mktcap["ric"], mktcap["quarter"]))

df["has_ric"] = df["ric"].notna()
df["has_sector"] = df["trbc_economic_sector"].notna()
df["has_price"] = df.apply(lambda r: (r["ric"], r["quarter"]) in price_keys if pd.notna(r["ric"]) else False, axis=1)
df["has_mktcap"] = df.apply(lambda r: (r["ric"], r["quarter"]) in mktcap_keys if pd.notna(r["ric"]) else False, axis=1)

n = len(df)
lines = []

def log(text=""):
    print(text)
    lines.append(text)

# Overall coverage
log("=" * 70)
log("DATA QUALITY SUMMARY")
log("=" * 70)
log(f"\nTotal holding rows: {n:,}")
log(f"Unique CUSIPs:     {df['cusip'].nunique():,}")
log(f"Unique RICs:       {df.loc[df['has_ric'], 'ric'].nunique():,}")
log(f"Funds:             {df['fund'].nunique()}")
log(f"Quarters:          {df['quarter'].nunique()}")

log(f"\n{'Metric':<30s} {'Missing':>10s} {'Coverage':>10s}")
log("-" * 52)
for label, col in [("RIC match", "has_ric"), ("TRBC sector", "has_sector"),
                    ("Quarterly price", "has_price"), ("Quarterly market cap", "has_mktcap")]:
    hits = df[col].sum()
    miss = n - hits
    log(f"{label:<30s} {miss:>10,d} {hits/n:>10.1%}")

# Top 20 unmatched CUSIPs
log(f"\n{'=' * 70}")
log("TOP 20 UNMATCHED CUSIPs (by frequency)")
log("=" * 70)

unmatched = df[~df["has_ric"]].copy()
top_unmatched = (
    unmatched.groupby("cusip")
    .agg(count=("cusip", "size"), issuer_name=("issuer_name", "first"))
    .sort_values("count", ascending=False)
    .head(20)
    .reset_index()
)

log(f"\n{'CUSIP':<12s} {'Count':>6s}  {'Issuer Name'}")
log("-" * 60)
for _, row in top_unmatched.iterrows():
    log(f"{row['cusip']:<12s} {row['count']:>6d}  {row['issuer_name']}")

# Coverage by fund
log(f"\n{'=' * 70}")
log("COVERAGE BY FUND")
log("=" * 70)

log(f"\n{'Fund':<30s} {'Holdings':>10s} {'RIC':>8s} {'Sector':>8s} {'Price':>8s} {'MktCap':>8s}")
log("-" * 74)
for fund, grp in df.groupby("fund"):
    fn = len(grp)
    log(f"{fund:<30s} {fn:>10,d} "
        f"{grp['has_ric'].mean():>8.1%} "
        f"{grp['has_sector'].mean():>8.1%} "
        f"{grp['has_price'].mean():>8.1%} "
        f"{grp['has_mktcap'].mean():>8.1%}")

# Coverage by time period
log(f"\n{'=' * 70}")
log("COVERAGE BY TIME PERIOD")
log("=" * 70)

df["year"] = df["quarter"].str[:4].astype(int)
bins = [
    ("2006-2010", 2006, 2010),
    ("2011-2015", 2011, 2015),
    ("2016-2020", 2016, 2020),
    ("2021-2025", 2021, 2025),
]

log(f"\n{'Period':<15s} {'Holdings':>10s} {'RIC':>8s} {'Sector':>8s} {'Price':>8s} {'MktCap':>8s}")
log("-" * 61)
for label, y_start, y_end in bins:
    period = df[(df["year"] >= y_start) & (df["year"] <= y_end)]
    pn = len(period)
    if pn == 0:
        continue
    log(f"{label:<15s} {pn:>10,d} "
        f"{period['has_ric'].mean():>8.1%} "
        f"{period['has_sector'].mean():>8.1%} "
        f"{period['has_price'].mean():>8.1%} "
        f"{period['has_mktcap'].mean():>8.1%}")

# Saving to file
notes_file = "../data/lseg/data_quality_notes.txt"
with open(notes_file, "w") as f:
    f.write("\n".join(lines))

print(f"\nSaved to {notes_file}")

DATA QUALITY SUMMARY

Total holding rows: 193,454
Unique CUSIPs:     11,974
Unique RICs:       9,554
Funds:             3
Quarters:          80

Metric                            Missing   Coverage
----------------------------------------------------
RIC match                          17,732      90.8%
TRBC sector                        24,420      87.4%
Quarterly price                    19,568      89.9%
Quarterly market cap               23,594      87.8%

TOP 20 UNMATCHED CUSIPs (by frequency)

CUSIP         Count  Issuer Name
------------------------------------------------------------
067901108       113  BARRICK GOLD CORP
09247X101       102  BLACKROCK INC
13645T100        99  CANADIAN PAC RY LTD
216648402        94  COOPER COS INC
512807108        85  LAM RESEARCH CORP
124857202        77  CBS CORP NEW
G5480U120        77  LIBERTY GLOBAL PLC
44267D107        76  HOWARD HUGHES CORP
38259P508        75  GOOGLE INC
708160106        68  PENNEY J C INC
G5480U104        67  LIBERTY G